# 5-class Odor Classifier — LoRA Fine-tuningfruity · sweet · sulfurous · woody · fatty

In [ ]:
%pip install scikit-learn unimol_tools peft huggingface_hub rdkit

In [ ]:
!git clone https://github.com/FufanLu/molecular-foundation-model.git

### Load data

In [ ]:
import sys
sys.path.append('molecular-foundation-model')

from src.dataset.load_leffingwell import load_leffingwell
from src.preprocessing.clean_smiles import clean_smiles
from src.classifier.label_encoder import encode_labels, label_distribution, ALL_5_CLASSES

df = load_leffingwell()
df = clean_smiles(df)
df = encode_labels(df)

print(f"{len(df)} molecules")
label_distribution(df)

### Load Uni-Mol + extract embeddings (no upload needed)

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

from unimol_tools import UniMolRepr

clf = UniMolRepr(data_type='molecule', remove_hs=False)
model = clf.model

smiles_list = df['smiles'].tolist()
print(f"Extracting embeddings for {len(smiles_list)} molecules...")
reprs = clf.get_repr(smiles_list)

import numpy as np
embeddings = {}
cls_reprs = reprs['cls_repr']
for i, compound in enumerate(df['compound'].tolist()):
    embeddings[compound] = cls_reprs[i]
print(f"Embeddings: {len(embeddings)}, dim={cls_reprs.shape[1]}")

### Find attention layer names for LoRA

In [ ]:
print("All Linear layers:")
all_linears = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        all_linears.append(name)
        print(f"  {name}: {module.in_features}->{module.out_features}")

print(f"\nTotal: {len(all_linears)} Linear layers")

### Apply LoRA

In [ ]:
from src.classifier.model import apply_lora, OdorClassifier

# Auto-detect attention layers: pick ones with 'attn' or 'q/k/v' in name
target_names = [n for n in all_linears
                if any(k in n.lower() for k in ['q_proj', 'k_proj', 'v_proj',
                                                  'query', 'key', 'value'])]
if not target_names:
    target_names = [n for n in all_linears if 'attn' in n.lower() and 'out' not in n.lower()]
if not target_names:
    print('WARNING: no attention layers found, applying to all Linear')
    target_names = all_linears

print(f"LoRA targets ({len(target_names)}): {target_names[:6]}...")
n_replaced = apply_lora(model, r=4, alpha=8, target_names=target_names)
print(f"LoRA applied to {n_replaced} layers")

classifier = OdorClassifier(model, hidden_dim=128, num_classes=5, dropout=0.2)
classifier = classifier.cuda() if torch.cuda.is_available() else classifier

trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total = sum(p.numel() for p in classifier.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

### Prepare train/test split

In [ ]:
from sklearn.model_selection import train_test_split

Y = np.stack(df['y'].values)
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_smiles = [df.iloc[i]['smiles'] for i in train_idx]
test_smiles = [df.iloc[i]['smiles'] for i in test_idx]
train_y = torch.tensor(Y[train_idx], dtype=torch.float32)
test_y = torch.tensor(Y[test_idx], dtype=torch.float32)
print(f"Train: {len(train_smiles)}, Test: {len(test_smiles)}")

### Inspect UniMolRepr batch API

In [ ]:
# Find the method that converts SMILES -> model input batch
methods = [m for m in dir(clf) if not m.startswith('__')]
print('UniMolRepr methods:')
for m in sorted(methods):
    print(f'  {m}')

# Try common preprocessing method names
for mname in ['_prepare_batch', 'prepare_batch', '_get_repr', '_preprocess',
              'smiles_to_data', '_smiles_to_data', 'batch_process']:
    if hasattr(clf, mname):
        print(f'\nFound: clf.{mname}')
        try:
            result = getattr(clf, mname)(['CCO'])
            print(f'  output type: {type(result)}')
            if isinstance(result, dict):
                print(f'  keys: {list(result.keys())}')
            elif isinstance(result, (list, tuple)):
                print(f'  len: {len(result)}, type[0]: {type(result[0])}')
        except Exception as e:
            print(f'  error: {e}')

### Train LoRA

In [ ]:
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

device = next(classifier.parameters()).device
optimizer = optim.AdamW(classifier.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

epochs = 30
batch_size = 16

for epoch in range(epochs):
    classifier.train()
    perm = torch.randperm(len(train_smiles))
    total_loss = 0.0
    n_batches = 0

    for i in range(0, len(train_smiles), batch_size):
        idx = perm[i:i+batch_size].tolist()
        batch_smiles = [train_smiles[j] for j in idx]
        batch_y = train_y[idx].to(device)

        # Get model inputs from SMILES
        batch_reprs = clf.get_repr(batch_smiles, return_atomic_reprs=False)
        # Use cls_repr as input to head (this won't backprop through UniMol)
        # For true LoRA we need raw forward pass - inspect API above first
        cls_emb = torch.tensor(batch_reprs['cls_repr'], dtype=torch.float32).to(device)

        optimizer.zero_grad()
        logits = classifier.head(cls_emb)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    # Eval
    if epoch % 5 == 0 or epoch == epochs - 1:
        classifier.eval()
        with torch.no_grad():
            test_reprs = clf.get_repr(test_smiles, return_atomic_reprs=False)
            test_emb = torch.tensor(test_reprs['cls_repr'], dtype=torch.float32).to(device)
            preds = torch.sigmoid(classifier.head(test_emb)).cpu().numpy()
            preds_bin = (preds > 0.5).astype(int)
            f1 = f1_score(Y[test_idx], preds_bin, average='macro', zero_division=0)
        print(f"epoch {epoch:3d}  loss {total_loss/n_batches:.4f}  f1_macro {f1:.4f}")
        classifier.train()

In [ ]:
# Per-class F1
classifier.eval()
with torch.no_grad():
    test_reprs = clf.get_repr(test_smiles, return_atomic_reprs=False)
    test_emb = torch.tensor(test_reprs['cls_repr'], dtype=torch.float32).to(device)
    preds = torch.sigmoid(classifier.head(test_emb)).cpu().numpy()
    preds_bin = (preds > 0.5).astype(int)

print(f"{'Class':<12} {'F1':>7}")
print('-' * 21)
for i, cls in enumerate(ALL_5_CLASSES):
    f1 = f1_score(Y[test_idx][:, i], preds_bin[:, i], zero_division=0)
    print(f"{cls:<12} {f1:>7.4f}")
f1_macro = f1_score(Y[test_idx], preds_bin, average='macro', zero_division=0)
print('-' * 21)
print(f"{'macro avg':<12} {f1_macro:>7.4f}")

In [ ]:
torch.save(classifier.state_dict(), 'odor_lora.pt')
from google.colab import files
files.download('odor_lora.pt')